# 🏭 SPATIOTEMPORAL CONGESTION-AWARE MULTI-AGENT SDVRP
## Research-Grade Warehouse Optimization Framework

This notebook implements the advanced Spatiotemporal Congestion-Aware Multi-Agent SDVRP algorithm, incorporating:
- Spatiotemporal (4D) congestion heatmap forecasting `H(x,y,t)`
- Multi-objective routing (Distance, Congestion, Wait Times, Imbalance, Overlap)
- Congestion-aware VNS local search


In [1]:
!pip install -q numpy matplotlib networkx ortools pandas



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


## Core Implementation


In [ ]:
# =====================================================================================
# SPATIOTEMPORAL CONGESTION-AWARE MULTI-AGENT SDVRP
# Research-Grade Warehouse Optimization Framework
# =====================================================================================
#
# FEATURES
# --------
# 1. Split Delivery Vehicle Routing Problem (SDVRP)
# 2. Spatiotemporal congestion heatmap H(x,y,t)
# 3. Predictive congestion-aware batching
# 4. Multi-objective optimization
# 5. Congestion-aware VNS local search
# 6. Route overlap minimization
# 7. Dynamic edge weighting
# 8. OR-Tools TSP optimization
#
# OBJECTIVE
# ---------
# Minimize:
#
# F = αD + βC + γW + δB + εO
#
# D = Travel Distance
# C = Congestion
# W = Waiting / Traffic Delay
# B = Workload Imbalance
# O = Route Overlap
#
# =====================================================================================

# =====================================================================================
# IMPORTS
# =====================================================================================

import math
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from typing import List, Tuple, Dict, Set

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

# =====================================================================================
# DATA MODELS
# =====================================================================================

Coord = Tuple[int, int]

@dataclass
class Product:
    sku: str
    x: int
    y: int
    category: str
    zone: str

@dataclass
class OrderItem:
    sku: str
    qty: int

@dataclass
class Order:
    order_id: str
    items: List[OrderItem]

@dataclass
class Config:
    max_batch_units: int = 25
    alpha_distance: float = 1.0
    beta_congestion: float = 4.0
    gamma_balance: float = 1.5
    epsilon_overlap: float = 3.0
    congestion_decay: float = 0.93
    time_horizon: int = 80

# =====================================================================================
# WAREHOUSE GRAPH
# =====================================================================================

class WarehouseGraph:

    def __init__(self, width, height, blocked):
        self.width = width
        self.height = height
        self.blocked = blocked

        self.G = nx.Graph()

        for x in range(width):
            for y in range(height):

                if (x, y) in blocked:
                    continue

                self.G.add_node((x, y))

                for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:

                    nx2 = x + dx
                    ny2 = y + dy

                    if (
                        0 <= nx2 < width and
                        0 <= ny2 < height and
                        (nx2, ny2) not in blocked
                    ):
                        self.G.add_edge((x,y), (nx2,ny2), weight=1)

        self.cache = {}

    def dist(self, a, b):

        if a == b:
            return 0

        key = (a,b)

        if key in self.cache:
            return self.cache[key]

        try:
            d = nx.astar_path_length(
                self.G,
                a,
                b,
                heuristic=lambda x,y: abs(x[0]-y[0]) + abs(x[1]-y[1]),
                weight="weight"
            )
        except:
            d = 9999

        self.cache[key] = d
        self.cache[(b,a)] = d

        return d

# =====================================================================================
# SPATIOTEMPORAL CONGESTION MODEL
# =====================================================================================

class SpatiotemporalHeatmap:

    def __init__(self, width, height, time_horizon, decay=0.95):

        self.width = width
        self.height = height
        self.time_horizon = time_horizon
        self.decay = decay

        self.H = np.zeros((width, height, time_horizon))

    def add_route(self, route, start_time=0, weight=1.0):

        t = start_time

        for c in route:

            if t >= self.time_horizon:
                break

            x, y = c

            self.H[x,y,t] += weight

            # spatial spread
            for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:

                nx2 = x + dx
                ny2 = y + dy

                if (
                    0 <= nx2 < self.width and
                    0 <= ny2 < self.height
                ):
                    self.H[nx2,ny2,t] += weight * 0.4

            t += 1

    def congestion(self, coord, t):

        if t >= self.time_horizon:
            t = self.time_horizon - 1

        x, y = coord

        return self.H[x,y,t]

    def decay_all(self):

        self.H *= self.decay

# =====================================================================================
# DATA GENERATION
# =====================================================================================

def build_warehouse(width=24, height=14):

    blocked = set()

    for x in range(2, width-2):
        for y in range(2, height-2):

            if x % 3 != 0 and y % 4 not in (0,3):
                blocked.add((x,y))

    return WarehouseGraph(width, height, blocked), blocked

# =====================================================================================

def generate_products(blocked, width, height, n_products=80):

    walkable = []

    for x in range(width):
        for y in range(height):

            if (x,y) not in blocked:

                for dx,dy in [(-1,0),(1,0),(0,-1),(0,1)]:

                    if (x+dx,y+dy) in blocked:
                        walkable.append((x,y))
                        break

    zones = ["A","B","C","D"]
    cats  = ["Bakery","Frozen","Snacks","Dairy"]

    random.shuffle(walkable)

    products = []

    for i in range(n_products):

        x, y = walkable[i]

        zi = i * len(zones) // n_products

        products.append(
            Product(
                sku=f"SKU-{i}",
                x=x,
                y=y,
                category=cats[zi],
                zone=zones[zi]
            )
        )

    return products

# =====================================================================================

def generate_orders(products, n_orders=60):

    orders = []

    zone_map = defaultdict(list)

    for p in products:
        zone_map[p.zone].append(p.sku)

    all_skus = [p.sku for p in products]

    for i in range(n_orders):

        zone = random.choice(list(zone_map.keys()))

        n_items = random.randint(1,4)

        chosen = random.sample(
            zone_map[zone] if random.random() < 0.75 else all_skus,
            n_items
        )

        items = []

        for sku in chosen:
            items.append(OrderItem(sku, random.randint(1,5)))

        orders.append(Order(f"O-{i}", items))

    return orders

# =====================================================================================
# LOOKUPS
# =====================================================================================

def build_lookup(products):

    sku_coord = {}
    sku_zone = {}

    for p in products:
        sku_coord[p.sku] = (p.x, p.y)
        sku_zone[p.sku] = p.zone

    return sku_coord, sku_zone

# =====================================================================================
# ROUTING
# =====================================================================================

def congestion_aware_route(
    grid,
    start,
    targets,
    heatmap,
    config,
    start_time=0
):

    unvisited = set(targets)

    route = [start]

    current = start

    total_cost = 0

    t = start_time

    while unvisited:

        best = None
        best_score = float("inf")

        for nxt in unvisited:

            dist = grid.dist(current, nxt)

            cong = heatmap.congestion(nxt, t)

            score = (
                config.alpha_distance * dist +
                config.beta_congestion * cong
            )

            if score < best_score:
                best_score = score
                best = nxt

        route.append(best)

        total_cost += best_score

        current = best

        unvisited.remove(best)

        t += 1

    route.append(start)

    return route, total_cost

# =====================================================================================
# ROUTE OVERLAP
# =====================================================================================

def overlap_penalty(route1, route2):

    s1 = set(route1)
    s2 = set(route2)

    return len(s1 & s2)

# =====================================================================================
# INITIAL SDVRP BATCHING
# =====================================================================================

def build_batches(
    orders,
    sku_coord,
    config,
    grid,
    heatmap,
    depot
):

    shelf_demand = defaultdict(int)

    for o in orders:
        for it in o.items:
            shelf_demand[sku_coord[it.sku]] += it.qty

    total = sum(shelf_demand.values())

    k = math.ceil(total / config.max_batch_units)

    batches = [[] for _ in range(k)]

    loads = [0] * k

    shelves = sorted(
        shelf_demand.keys(),
        key=lambda c: grid.dist(depot, c),
        reverse=True
    )

    for shelf in shelves:

        rem = shelf_demand[shelf]

        while rem > 0:

            best_batch = -1
            best_score = float("inf")

            for bi in range(len(batches)):

                cap = config.max_batch_units - loads[bi]

                if cap <= 0:
                    continue

                current_coords = [c for c,q in batches[bi]]

                route, dist_cost = congestion_aware_route(
                    grid,
                    depot,
                    current_coords + [shelf],
                    heatmap,
                    config
                )

                cong = sum(
                    heatmap.congestion(c, t)
                    for t,c in enumerate(route)
                )

                balance = loads[bi] / config.max_batch_units

                score = (
                    config.alpha_distance * dist_cost +
                    config.beta_congestion * cong +
                    config.gamma_balance * balance
                )

                if score < best_score:
                    best_score = score
                    best_batch = bi

            q = min(rem, config.max_batch_units - loads[best_batch])

            batches[best_batch].append((shelf, q))

            loads[best_batch] += q

            rem -= q

    return batches

# =====================================================================================
# CONGESTION-AWARE VNS
# =====================================================================================

def vns_optimize(
    batches,
    grid,
    depot,
    heatmap,
    config
):

    improved = True

    while improved:

        improved = False

        for i in range(len(batches)):

            for j in range(i+1, len(batches)):

                for ai in range(len(batches[i])):

                    for aj in range(len(batches[j])):

                        ci, qi = batches[i][ai]
                        cj, qj = batches[j][aj]

                        r1_before, c1_before = congestion_aware_route(
                            grid,
                            depot,
                            [c for c,q in batches[i]],
                            heatmap,
                            config
                        )

                        r2_before, c2_before = congestion_aware_route(
                            grid,
                            depot,
                            [c for c,q in batches[j]],
                            heatmap,
                            config
                        )

                        overlap_before = overlap_penalty(r1_before, r2_before)

                        temp_i = batches[i].copy()
                        temp_j = batches[j].copy()

                        temp_i[ai] = (cj, qj)
                        temp_j[aj] = (ci, qi)

                        r1_after, c1_after = congestion_aware_route(
                            grid,
                            depot,
                            [c for c,q in temp_i],
                            heatmap,
                            config
                        )

                        r2_after, c2_after = congestion_aware_route(
                            grid,
                            depot,
                            [c for c,q in temp_j],
                            heatmap,
                            config
                        )

                        overlap_after = overlap_penalty(r1_after, r2_after)

                        before = (
                            c1_before +
                            c2_before +
                            config.epsilon_overlap * overlap_before
                        )

                        after = (
                            c1_after +
                            c2_after +
                            config.epsilon_overlap * overlap_after
                        )

                        if after < before:

                            batches[i] = temp_i
                            batches[j] = temp_j

                            improved = True

    return batches

# =====================================================================================
# VISUALIZATION
# =====================================================================================

def plot_routes(
    grid,
    blocked,
    products,
    batches,
    depot,
    heatmap,
    config
):

    fig, ax = plt.subplots(figsize=(16,10))

    for (x,y) in blocked:
        ax.add_patch(
            plt.Rectangle((x-0.5,y-0.5),1,1,alpha=0.5)
        )

    cmap = plt.colormaps["tab10"]

    for bi, batch in enumerate(batches):

        coords = [c for c,q in batch]

        route, cost = congestion_aware_route(
            grid,
            depot,
            coords,
            heatmap,
            config
        )

        xs = [c[0] for c in route]
        ys = [c[1] for c in route]

        ax.plot(xs, ys, "-o", linewidth=2)

    ax.plot(depot[0], depot[1], "*", markersize=20)

    ax.set_title("Spatiotemporal Congestion-Aware SDVRP")

    plt.tight_layout()
    plt.savefig("spatiotemporal_routing.png", dpi=150)
    plt.show()

# =====================================================================================
# MAIN EXPERIMENT
# =====================================================================================

def run_experiment():
    print("=" * 80)
    print(" SPATIOTEMPORAL CONGESTION-AWARE SDVRP")
    print("=" * 80)

    config = Config()

    grid, blocked = build_warehouse()

    products = generate_products(blocked, 24, 14)

    orders = generate_orders(products)

    sku_coord, sku_zone = build_lookup(products)

    depot = (0,0)

    heatmap = SpatiotemporalHeatmap(
        24,
        14,
        config.time_horizon,
        config.congestion_decay
    )

    # =====================================================================================
    # BUILD INITIAL BATCHES
    # =====================================================================================

    print("\nBuilding congestion-aware SDVRP batches...")
    t0 = time.time()
    batches = build_batches(
        orders,
        sku_coord,
        config,
        grid,
        heatmap,
        depot
    )
    t1 = time.time()
    print(f"-> Formed {len(batches)} initial routes in {(t1-t0):.2f}s")

    # =====================================================================================
    # UPDATE HEATMAP
    # =====================================================================================

    for batch in batches:

        coords = [c for c,q in batch]

        route, _ = congestion_aware_route(
            grid,
            depot,
            coords,
            heatmap,
            config
        )

        heatmap.add_route(route)

    # =====================================================================================
    # VNS OPTIMIZATION
    # =====================================================================================

    print("\nRunning congestion-aware VNS optimization...")
    t2 = time.time()
    batches = vns_optimize(
        batches,
        grid,
        depot,
        heatmap,
        config
    )
    t3 = time.time()
    print(f"-> VNS complete in {(t3-t2):.2f}s")

    # =====================================================================================
    # FINAL EVALUATION
    # =====================================================================================

    total_distance = 0

    for batch in batches:

        coords = [c for c,q in batch]

        route, cost = congestion_aware_route(
            grid,
            depot,
            coords,
            heatmap,
            config
        )

        total_distance += cost

    print("\n================================================")
    print("FINAL RESULTS")
    print("================================================")
    print("Total Batches:", len(batches))
    print("Total Routing Cost (Distance + Congestion Penalty):", round(total_distance,2))
    
    print("\nGenerating Visualization...")
    plot_routes(
        grid,
        blocked,
        products,
        batches,
        depot,
        heatmap,
        config
    )
    print("Saved plot to spatiotemporal_routing.png")



ModuleNotFoundError: No module named 'matplotlib'

## Run the Experiment


In [ ]:
run_experiment()
